# LeetCode #1246: Palindrome Removal

https://leetcode.com/problems/palindrome-removal/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^n)$ | $O(n)$ |
| **Optimal: Interval DP ★** | $O(n^3)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force
Try every possible palindrome removal at each step recursively, branching on all subsets. Exponential state space makes this infeasible for $n > 20$.

### Optimal: Interval DP ★
`dp[i][j]` = minimum moves to remove `arr[i..j]`. Base cases: `dp[i][i] = 1` (single element), `dp[i][i+1] = 1` if equal else 2. Transition: if `arr[i] == arr[j]`, we can merge their removal — `dp[i][j] = dp[i+1][j-1]` (or 1 if they're adjacent). Also try every split point `k` and take `dp[i][k] + dp[k+1][j]`.

**Constraints:**
* `1 <= arr.length <= 100`
* `1 <= arr[i] <= 20`

## Solutions

### C#

In [ ]:
public class Solution {
    public int MinimumMoves(int[] arr) {
        int n = arr.Length;
        int[,] dp = new int[n, n];

        // Base case: every single element costs 1 removal
        for (int i = 0; i < n; i++) dp[i, i] = 1;

        // Fill by increasing interval length
        for (int len = 2; len <= n; len++) {
            for (int i = 0; i <= n - len; i++) {
                int j = i + len - 1;
                // Equal endpoints can be removed in the same move as the interior
                if (arr[i] == arr[j])
                    dp[i, j] = len == 2 ? 1 : dp[i + 1, j - 1];
                else
                    dp[i, j] = int.MaxValue;

                // Try every split; take the minimum across partitions
                for (int k = i; k < j; k++)
                    dp[i, j] = Math.Min(dp[i, j], dp[i, k] + dp[k + 1, j]);
            }
        }
        return dp[0, n - 1];
    }
}

### Python

In [ ]:
class Solution:
    def minimumMoves(self, arr: list[int]) -> int:
        n = len(arr)
        dp = [[0] * n for _ in range(n)]

        # Single-element intervals always cost 1 removal
        for i in range(n):
            dp[i][i] = 1

        for length in range(2, n + 1):
            for i in range(n - length + 1):
                j = i + length - 1
                # Equal endpoints can be removed alongside the inner palindrome
                if arr[i] == arr[j]:
                    dp[i][j] = 1 if length == 2 else dp[i + 1][j - 1]
                else:
                    dp[i][j] = float('inf')

                # Partition at every internal split and take minimum
                for k in range(i, j):
                    dp[i][j] = min(dp[i][j], dp[i][k] + dp[k + 1][j])

        return dp[0][n - 1]

### Go

In [ ]:
func minimumMoves(arr []int) int {
    n := len(arr)
    dp := make([][]int, n)
    for i := range dp {
        dp[i] = make([]int, n)
        dp[i][i] = 1 // single element: one removal
    }

    for length := 2; length <= n; length++ {
        for i := 0; i <= n-length; i++ {
            j := i + length - 1
            if arr[i] == arr[j] {
                // Equal endpoints merge into the interior's removal
                if length == 2 {
                    dp[i][j] = 1
                } else {
                    dp[i][j] = dp[i+1][j-1]
                }
            } else {
                dp[i][j] = 1<<31 - 1
            }
            // Check every partition to find the minimum
            for k := i; k < j; k++ {
                if v := dp[i][k] + dp[k+1][j]; v < dp[i][j] {
                    dp[i][j] = v
                }
            }
        }
    }
    return dp[0][n-1]
}

### Rust

In [ ]:
impl Solution {
    pub fn minimum_moves(arr: Vec<i32>) -> i32 {
        let n = arr.len();
        let mut dp = vec![vec![0i32; n]; n];

        // Base: removing one element takes one step
        for i in 0..n { dp[i][i] = 1; }

        for length in 2..=n {
            for i in 0..=(n - length) {
                let j = i + length - 1;
                if arr[i] == arr[j] {
                    // Equal endpoints reduce to the interior sub-problem
                    dp[i][j] = if length == 2 { 1 } else { dp[i + 1][j - 1] };
                } else {
                    dp[i][j] = i32::MAX / 2;
                }
                // Every split to find minimum total moves
                for k in i..j {
                    dp[i][j] = dp[i][j].min(dp[i][k] + dp[k + 1][j]);
                }
            }
        }
        dp[0][n - 1]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `arr = [1,3,4,1,5]`
`dp[0][3]`: `arr[0]==arr[3]==1`, so `dp[0][3] = dp[1][2] = 2`. Full: `dp[0][4] = min(dp[0][3]+dp[4][4], ...) = 3`.

### 2. Slightly Complex
**Input:** `arr = [1,2,1]`
`arr[0]==arr[2]==1`; `dp[0][2] = dp[1][1] = 1`. Remove the entire array in one palindrome step — answer **1**.

### 3. Edge Case: Time Factor
**Input:** `arr` of length 100, all distinct values.
The $O(n^3)$ loop runs $\approx 10^6$ iterations — each an $O(1)$ comparison. Time budget is comfortably met.

### 4. Edge Case: Space Factor
**Input:** `arr` of length 100.
`dp` is a $100 \times 100$ table — exactly 10,000 integers, $O(n^2)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `arr = [1,1,1,...,1]` (100 ones).
Every interval `[i,j]` is a palindrome; `dp[0][99] = 1` because the entire array is one palindrome removed in a single step.